# Another copy of gemini_audio_probe_pipeline.ipynb

Gemini audio-and-text evaluation using fixed prompts.

Set the Gemini environment variables and configure the audio, text, and fixed-prompt paths.

Use the setup steps in the repository README before running this notebook.


In [ ]:
!pip install -q openai httpx pandas tqdm librosa soundfile

In [ ]:
# ============================================================
# GEMINI BASELINE DEVIATION RUN
# FULL VERSION WITH TASK-AWARE PROMPTS
# ============================================================

import os
import re
import json
import time
import random
import base64
import librosa
import httpx
import pandas as pd

from tqdm import tqdm
from openai import OpenAI

# ============================================================
# CONFIG
# ============================================================

BASE_DIR = "/content/drive/MyDrive/probing"

SAVE_DIR = "/content/drive/MyDrive/probing/full_probe/albe"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_NAME = "gemini-2.5-flash"

# ============================================================
# GEMINI CLIENT
# ============================================================
GEMINI_API_KEY = API_KEY = os.environ["GEMINI_API_KEY"]
#http_client = httpx.Client(
#    proxy="http://127.0.0.1:7890"
#)

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url=os.environ["GEMINI_BASE_URL"],
   # http_client=http_client
)

# ============================================================
# PROMPTS
# ============================================================

# Delivery
SYSTEM_PROMPT_DELIVERY = """
You are a speech emotion classifier.

Determine whether the speaker sounds neutral or emotional.

Choices:
[neutral, emotional]

Return ONLY valid JSON:
{"label": "<neutral or emotional>", "confidence": <0.0-1.0>}
"""

USER_PROMPT_DELIVERY = "Classify the speech."

# Intensity
def build_intensity_prompt(target_emotion, related_emotion=None):

    if related_emotion:
        choices = f"[neutral, {target_emotion}, {related_emotion}]"
    else:
        choices = f"[neutral, {target_emotion}]"

    return f"""
You are a speech emotion classifier.

Determine the emotion of the speaker.

Choices:
{choices}

Return ONLY valid JSON:
{{"label": "<...>", "confidence": <0.0-1.0>}}
"""

USER_PROMPT_INTENSITY = "Classify the emotion of the speech."

In [ ]:
BASE_DIR = "/content/drive/MyDrive/probing"
#SAVE_DIR = "/content/drive/MyDrive/probing/baseline_outputs_edited"

SAVE_DIR = "/content/drive/MyDrive/probing/full_probe/close"


In [ ]:
from openai import OpenAI

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url=os.environ["GEMINI_BASE_URL"],
)

response = client.chat.completions.create(
    model="gemini-3.8-flash",
    messages=[
        {"role": "user", "content": "Reply with exactly: GEMINI_OK"}
    ],
    temperature=0,
)

answer = response.choices[0].message.content.strip()
print("Response:", answer)
assert "GEMINI_OK" in answer, f"Unexpected response: {answer}"
print("✅ Gemini text API is usable.")

In [ ]:
CLUSTERS = [
    ["anger", "annoyance", "disapproval"],
    ["disgust", "disapproval"],
    ["fear", "nervousness"],
    ["joy", "amusement", "excitement", "pride", "optimism", "love", "admiration", "relief", "gratitude", "caring", "desire"],
    ["confusion", "curiosity", "realization", "surprise"],
    ["sadness", "disappointment", "remorse", "grief"],
    ["surprise", "realization"]
]
emotion_to_cluster = {}

for cluster in CLUSTERS:
    for e in cluster:
        emotion_to_cluster[e] = cluster
import random
def sample_target_emotion(emotion_list):

    if isinstance(emotion_list, str):
        # if stored as string like "['a','b']"
        import ast
        emotion_list = ast.literal_eval(emotion_list)


    if not emotion_list:
        raise ValueError("Empty emotion list")

    return random.choice(emotion_list)

def pick_distractor(target):

    cluster = emotion_to_cluster.get(target, None)
    print(cluster)

    # Prefer same-cluster distractor
    if cluster:
        candidates = [e for e in cluster if e != target]
        if candidates:
            return random.choice(candidates)

    # fallback (rare)
    all_emotions = list(emotion_to_cluster.keys())
    candidates = [e for e in all_emotions if e != target]
    return random.choice(candidates)
def build_subtle_prompt(target_emotion):
    target = sample_target_emotion(target_emotion)
    print(target)

    distractor = pick_distractor(target_emotion)
    print(distractor)

    choices = [
        target,
        distractor,
        "neutral"
    ]

    random.shuffle(choices)

    choice_str = "\n".join(
        [f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]
    )


    system_prompt = """
You are a speech emotion classifier.

Choose the most appropriate fine-grained emotion expressed in the speaker’s voice.

You must choose ONLY ONE option.

Return ONLY valid JSON:
{"label": "<one of the choices>", "confidence": <0.0-1.0>}
"""

    user_prompt = f"""
Which of the following best describes the speaker’s emotion?

Choices:
{choice_str}
"""

    return system_prompt, user_prompt

In [ ]:
CLUSTERS = [
    ["anger", "annoyance", "disapproval"],
    ["disgust", "disapproval"],
    ["fear", "nervousness"],
    ["joy", "amusement", "excitement", "pride", "optimism", "love", "admiration", "relief", "gratitude", "caring", "desire"],
    ["confusion", "curiosity", "realization", "surprise"],
    ["sadness", "disappointment", "remorse", "grief"],
    ["surprise", "realization"]
]
emotion_to_cluster = {}

for cluster in CLUSTERS:
    for e in cluster:
        emotion_to_cluster[e] = cluster
import random
def sample_target_emotion(emotion_list):

    if isinstance(emotion_list, str):
        # if stored as string like "['a','b']"
        import ast
        emotion_list = ast.literal_eval(emotion_list)


    if not emotion_list:
        raise ValueError("Empty emotion list")

    return random.choice(emotion_list)

def pick_distractor(target):

    cluster = emotion_to_cluster.get(target, None)
    print(cluster)

    # Prefer same-cluster distractor
    if cluster:
        candidates = [e for e in cluster if e != target]
        if candidates:
            return random.choice(candidates)

    # fallback (rare)
    all_emotions = list(emotion_to_cluster.keys())
    candidates = [e for e in all_emotions if e != target]
    return random.choice(candidates)
def build_subtle_prompt(target_emotion):
    target = sample_target_emotion(target_emotion)
    print(target)

    distractor = pick_distractor(target_emotion)
    print(distractor)

    choices = [
        target,
        distractor,
        "neutral"
    ]

    random.shuffle(choices)

    choice_str = "\n".join(
        [f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]
    )


    system_prompt = """
You are a speech emotion classifier.

Choose the most appropriate fine-grained emotion expressed in the speaker’s voice.

You must choose ONLY ONE option.

Return ONLY valid JSON:
{"label": "<one of the choices>", "confidence": <0.0-1.0>}
"""

    user_prompt = f"""
Which of the following best describes the speaker’s emotion?

Choices:
{choice_str}
"""

    return system_prompt, user_prompt

In [ ]:
# ============================================================
# MEMORY
# ============================================================

import gc

def clear_memory():
    gc.collect()

# ============================================================
# PATH REMAP
# ============================================================

def remap_to_colab_path(old_path):

    if not isinstance(old_path, str):
        raise ValueError(f"Invalid path: {old_path}")

    # normalize Windows → Unix
    old_path = old_path.replace("\\", "/").strip()

    # join with Drive base
    new_path = os.path.join(BASE_DIR, old_path)

    return new_path

# ============================================================
# JSON SAFE PARSER
# ============================================================

def _parse_valid_json(text):

    obj = json.loads(text)

    label = obj.get("label", None)
    confidence = obj.get("confidence", None)

    return label, confidence


def parse_json_safe(text):

    if not isinstance(text, str):
        return None, None

    raw = text.strip()

    # ---- remove markdown fences ----
    raw = re.sub(r"^```json", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"^```", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()

    # ---- strict JSON first ----
    try:
        return _parse_valid_json(raw)
    except:
        pass

    # ---- extract JSON substring ----
    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1:

        json_str = raw[start:end+1]

        # fix common Gemini formatting issues
        json_str = re.sub(r"(\w+):", r'"\1":', json_str)
        json_str = json_str.replace("'", '"')
        json_str = re.sub(r",\s*}", "}", json_str)

        try:
            return _parse_valid_json(json_str)
        except:
            pass

    # ---- regex fallback ----
    label = None
    confidence = None

    label_match = re.search(
        r"(neutral|emotional|happy|sad|angry|fearful|disgust|caring|supportive|curious)",
        raw.lower()
    )

    if label_match:
        label = label_match.group(1)

    conf_match = re.search(r"(0\.\d+|1\.0+)", raw)

    if conf_match:
        confidence = float(conf_match.group(1))

    return label, confidence

# ============================================================
# PROMPT ROUTER
# ============================================================

def extract_emotion_from_filename(path):

    filename = os.path.basename(path)

    parts = filename.split("_")

    if len(parts) < 3:
        return None

    code = parts[2]

    mapping = {
        "ANG": "angry",
        "DIS": "disgust",
        "FEA": "fearful",
        "HAP": "happy",
        "SAD": "sad",
        "NEU": "neutral"
    }

    return mapping.get(code, None)


def get_prompt_for_row(row):

    task = row["task"]

    if task == "style":

        return SYSTEM_PROMPT_DELIVERY, USER_PROMPT_DELIVERY

    elif task == "intensity":

        target = extract_emotion_from_filename(
            row["neutral_path"]
        )

        related = row.get("related_emotion", None)

        system_prompt = build_intensity_prompt(
            target,
            related
        )

        return system_prompt, USER_PROMPT_INTENSITY

    else:
        raise ValueError(f"Unknown task type: {task}")

# ============================================================
# AUDIO ENCODING
# ============================================================

def safe_b64encode(audio_path):

    try:

        with open(audio_path, "rb") as f:
            audio_bytes = f.read()

        return base64.b64encode(
            audio_bytes
        ).decode("utf-8")

    except Exception as e:

        print(f"❌ Audio encoding error: {e}")

        return None

# ============================================================
# GEMINI INFERENCE
# ============================================================

def judge_gemini_audio(
    wav_path,
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_retries=3
):

    base64_audio = safe_b64encode(wav_path)

    if base64_audio is None:
        return None, None, None

    content = [
        {
            "type": "text",
            "text": user_prompt
        },
        {
            "type": "input_audio",
            "input_audio": {
                "data": base64_audio,
                "format": "wav"
            }
        }
    ]

    for attempt in range(max_retries):

        try:

            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=temperature,
                reasoning_effort="low",
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": content
                    }
                ]
            )

            raw_response = (
                response.choices[0]
                .message
                .content
                .strip()
            )

            label, confidence = parse_json_safe(
                raw_response
            )

            return (
                label,
                confidence,
                raw_response
            )

        except Exception as e:

            print(
                f"❌ Gemini error "
                f"(attempt {attempt+1}): {e}"
            )

            time.sleep(2)

    return None, None, None

# ============================================================
# SAMPLE SUBSET PER TASK
# ============================================================

def sample_subset(paths, fraction):

    n = max(10, int(len(paths) * fraction))

    return random.sample(
        paths,
        min(n, len(paths))
    )

# ============================================================
# MAIN BASELINE RUNNER
# ============================================================

def run_baseline_deviation_gemini(metadata):

    rows = []

    grouped = metadata.groupby("baseline_type")

    for baseline_type, group_df in grouped:

        print(
            f"\n===== Running baseline group: "
            f"{baseline_type} ====="
        )

        unique_rows = group_df.drop_duplicates(
            subset=["neutral_path"]
        )

        temp0_rows = unique_rows.sample(
            n=max(10, int(len(unique_rows) * 0.10)),
            random_state=42
        )

        temp03_rows = unique_rows.sample(
            n=max(10, int(len(unique_rows) * 0.30)),
            random_state=42
        )

        # ====================================================
        # TEMP = 0
        # ====================================================

        for _, row in tqdm(
            temp0_rows.iterrows(),
            total=len(temp0_rows)
        ):

            wav_path = remap_to_colab_path(
                row["neutral_path"]
            )

            if not os.path.exists(wav_path):
                continue

            system_prompt, user_prompt = (
                get_prompt_for_row(row)
            )

            for run_idx in range(2):

                label, conf, raw = judge_gemini_audio(
                    wav_path,
                    system_prompt,
                    user_prompt,
                    temperature=0.0
                )

                rows.append({
                    "id": row["id"],
                    "audio": wav_path,
                    "baseline_type": baseline_type,
                    "temperature": 0.0,
                    "run": run_idx + 1,
                    "label": label,
                    "confidence": conf,
                    "raw_response": raw,
                    "model": "gemini_2_5_flash"
                })

                clear_memory()

        # ====================================================
        # TEMP = 0.3
        # ====================================================

        for _, row in tqdm(
            temp03_rows.iterrows(),
            total=len(temp03_rows)
        ):

            wav_path = remap_to_colab_path(
                row["neutral_path"]
            )

            if not os.path.exists(wav_path):
                continue

            system_prompt, user_prompt = (
                get_prompt_for_row(row)
            )

            for run_idx in range(3):

                label, conf, raw = judge_gemini_audio(
                    wav_path,
                    system_prompt,
                    user_prompt,
                    temperature=0.3
                )

                rows.append({
                    "id": row["id"],
                    "audio": wav_path,
                    "baseline_type": baseline_type,
                    "temperature": 0.3,
                    "run": run_idx + 1,
                    "label": label,
                    "confidence": conf,
                    "raw_response": raw,
                    "model": "gemini_2_5_flash"
                })

                clear_memory()

    df = pd.DataFrame(rows)

    csv_path = os.path.join(
        SAVE_DIR,
        "gemini_baseline_results.csv"
    )

    df.to_csv(csv_path, index=False)

    print(f"\nSaved CSV: {csv_path}")

In [ ]:
# ============================================================
# NEUTRAL PASS
# ============================================================

def run_neutral_pass(metadata):

    results = {}
    rows = []

    unique_neutral = (
        metadata["neutral_path"]
        .drop_duplicates()
    )

    print(
        f"\n===== Running NEUTRAL pass "
        f"({len(unique_neutral)} files) ====="
    )

    for path in tqdm(unique_neutral):

        wav_path = remap_to_colab_path(path)

        if not os.path.exists(wav_path):

            print(f"⚠️ Missing: {wav_path}")
            continue

        row = metadata[
            metadata["neutral_path"] == path
        ].iloc[0]

        system_prompt, user_prompt = (
            get_prompt_for_row(row)
        )

        try:

            label, conf, raw = judge_gemini_audio(
                wav_path,
                system_prompt,
                user_prompt,
                temperature=0.0
            )

        except Exception as e:

            print(f"❌ Error: {wav_path} | {e}")
            continue

        results[path] = {
            "label": label,
            "confidence": conf,
            "raw": raw
        }

        rows.append({
            "neutral_path": path,
            "label": label,
            "confidence": conf,
            "raw_response": raw,
            "model": "gemini_2_5_flash"
        })

        clear_memory()

    # ========================================================
    # SAVE CSV
    # ========================================================

    df = pd.DataFrame(rows)

    csv_path = os.path.join(
        SAVE_DIR,
        "neutral_results.csv"
    )

    df.to_csv(csv_path, index=False)

    print(f"Saved neutral CSV: {csv_path}")

    return results

In [ ]:
# ============================================================
# MANIPULATED PASS
# ============================================================

def run_manipulated_pass(metadata):

    rows = []

    print(
        f"\n===== Running MANIPULATED pass "
        f"({len(metadata)} samples) ====="
    )

    for _, row in tqdm(
        metadata.iterrows(),
        total=len(metadata)
    ):

        wav_path = remap_to_colab_path(
            row["manipulated_path"]
        )

        if not os.path.exists(wav_path):

            print(f"⚠️ Missing: {wav_path}")
            continue

        system_prompt, user_prompt = (
            get_prompt_for_row(row)
        )

        try:

            label, conf, raw = judge_gemini_audio(
                wav_path,
                system_prompt,
                user_prompt,
                temperature=0.0
            )

        except Exception as e:

            print(f"❌ Error: {wav_path} | {e}")
            continue

        rows.append({
            "id": row["id"],
            "task": row["task"],
            "neutral_path": row["neutral_path"],
            "manipulated_path": row["manipulated_path"],
            "intensity": row.get("intensity", None),
            "condition": row.get("condition", None),
            "baseline_type": row.get("baseline_type", None),
            "label": label,
            "confidence": conf,
            "raw_response": raw,
            "model": "gemini_2_5_flash"
        })

        clear_memory()

    # ========================================================
    # SAVE CSV
    # ========================================================

    df = pd.DataFrame(rows)

    csv_path = os.path.join(
        SAVE_DIR,
        "manipulated_results.csv"
    )

    df.to_csv(csv_path, index=False)

    print(f"Saved manipulated CSV: {csv_path}")

    return df

In [ ]:
# ============================================================
# MERGE NEUTRAL + MANIPULATED RESULTS
# ============================================================

def merge_and_save(
    manip_df,
    neutral_results
):

    merged_rows = []

    for _, row in manip_df.iterrows():

        neutral_info = neutral_results.get(
            row["neutral_path"],
            None
        )

        if neutral_info is None:
            continue

        merged_rows.append({

            # =================================================
            # METADATA
            # =================================================

            "id": row["id"],
            "task": row["task"],

            "neutral_path": row["neutral_path"],
            "manipulated_path": row["manipulated_path"],

            "intensity": row.get("intensity", None),
            "condition": row.get("condition", None),
            "baseline_type": row.get("baseline_type", None),

            # =================================================
            # MANIPULATED OUTPUT
            # =================================================

            "label": row["label"],
            "confidence": row["confidence"],
            "raw_response": row["raw_response"],

            # =================================================
            # NEUTRAL OUTPUT
            # =================================================

            "neutral_label": neutral_info["label"],
            "neutral_confidence": neutral_info["confidence"],
            "neutral_raw": neutral_info["raw"],

            # =================================================
            # MODEL
            # =================================================

            "model": "gemini_2_5_flash"
        })

    # ========================================================
    # SAVE CSV
    # ========================================================

    df = pd.DataFrame(merged_rows)

    csv_path = os.path.join(
        SAVE_DIR,
        "merged_results.csv"
    )

    df.to_csv(csv_path, index=False)

    print(f"Saved merged CSV: {csv_path}")

    return df

In [ ]:
# =========================================================
# RUN PIPELINE
# =========================================================
metadata = pd.read_csv("/content/drive/MyDrive/probing/subset_data.csv")
metadata = metadata[metadata["baseline_type"] != 'subtle']
neutral_results = run_neutral_pass(metadata)
manip_df = run_manipulated_pass(metadata)
merged_df = merge_and_save(manip_df, neutral_results)
print("Pipeline complete.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# GEMINI TEXT-ONLY INFERENCE
# ============================================================

def judge_gemini_text(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_retries=3
):

    for attempt in range(max_retries):

        try:

            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=temperature,
                reasoning_effort="low",
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ]
            )

            raw_response = (
                response.choices[0]
                .message
                .content
                .strip()
            )

            label, confidence = parse_json_safe(
                raw_response
            )

            return (
                label,
                confidence,
                raw_response
            )

        except Exception as e:

            print(
                f"❌ Gemini text error "
                f"(attempt {attempt+1}): {e}"
            )

            time.sleep(2)

    return None, None, None

# ============================================================
# GEMINI AUDIO INFERENCE
# ============================================================

def safe_b64encode(audio_path):

    try:

        with open(audio_path, "rb") as f:
            audio_bytes = f.read()

        return base64.b64encode(
            audio_bytes
        ).decode("utf-8")

    except Exception as e:

        print(f"❌ Audio encoding error: {e}")

        return None


def judge_gemini_audio(
    wav_path,
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_retries=3
):

    base64_audio = safe_b64encode(
        wav_path
    )

    if base64_audio is None:
        return None, None, None

    content = [
        {
            "type": "text",
            "text": user_prompt
        },
        {
            "type": "input_audio",
            "input_audio": {
                "data": base64_audio,
                "format": "wav"
            }
        }
    ]

    for attempt in range(max_retries):

        try:

            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=temperature,
                reasoning_effort="low",
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": content
                    }
                ]
            )

            raw_response = (
                response.choices[0]
                .message
                .content
                .strip()
            )

            label, confidence = parse_json_safe(
                raw_response
            )

            return (
                label,
                confidence,
                raw_response
            )

        except Exception as e:

            print(
                f"❌ Gemini audio error "
                f"(attempt {attempt+1}): {e}"
            )

            time.sleep(2)

    return None, None, None

In [ ]:
# ============================================================
# AUDIO+TEXT RUN (FIXED + TRANSCRIPT MERGE VERSION)
# ============================================================

import os
import gc
import json
import pandas as pd

from tqdm import tqdm

# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/subprobe"

FIXED_PROMPT_PATH = os.path.join(
    BASE_DIR,
    "fixed_prompt_subset_1000.csv"
)

TEXT_SOURCE_PATH = os.path.join(
    BASE_DIR,
    "fixed_subset_1000_with_text.csv"
)

SAVE_DIR = os.path.join(
    BASE_DIR,
    "task3_same_cluster_gemini_multimodal_fixed"
)

os.makedirs(SAVE_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    SAVE_DIR,
    "audio_text_results.csv"
)

# ============================================================
# CLEANUP
# ============================================================

def cleanup():
    gc.collect()

# ============================================================
# PATH NORMALIZER
# ============================================================

def normalize_path(p):

    p = str(p).replace("\\", "/").strip()

    p = p.replace(
        "/content/drive/MyDrive/subprobe/",
        ""
    )

    return p.lower()

# ============================================================
# RESOLVE PATH
# ============================================================

def resolve_path(p):

    p = str(p).replace("\\", "/").strip()

    if p.startswith("/content/"):
        return p

    return os.path.join(BASE_DIR, p)

# ============================================================
# LOAD DATA
# ============================================================

prompt_df = pd.read_csv(
    FIXED_PROMPT_PATH
)

text_df = pd.read_csv(
    TEXT_SOURCE_PATH
)

# normalize
prompt_df["audio_path_norm"] = (
    prompt_df["audio_path"]
    .apply(normalize_path)
)

text_df["audio_path_norm"] = (
    text_df["audio_path"]
    .apply(normalize_path)
)

# ============================================================
# SAFE MERGE
# ============================================================

merged_df = prompt_df.merge(

    text_df[
        [
            "audio_path_norm",
            "Utterance"
        ]
    ],

    on="audio_path_norm",
    how="left",
    validate="many_to_one"
)

print(
    "Merged rows:",
    len(merged_df)
)

print(
    "Missing transcript rows:",
    merged_df["Utterance"].isna().sum()
)

# ============================================================
# AUDIO+TEXT PROMPT
# ============================================================

def build_audio_text_prompt(
    transcript,
    user_prompt
):

    return f"""
Transcript:
{transcript}

Use BOTH the speech audio and transcript.

{user_prompt}
"""

# ============================================================
# BUILD FIXED PROMPT
# ============================================================

def build_fixed_prompt(row):

    choices = json.loads(
        row["choices"]
    )

    choice_str = "\n".join([

        f"{chr(65+i)}. {c}"

        for i, c in enumerate(choices)
    ])

    system_prompt = """
You are a speech emotion classifier.

Choose the most appropriate fine-grained emotion expressed by the speaker.

You must choose ONLY ONE option.

Return ONLY valid JSON:
{"label": "<one of the choices>", "confidence": <0.0-1.0>}
"""

    user_prompt = f"""
Which of the following best describes the speaker's emotion?

Choices:
{choice_str}
"""

    return (
        system_prompt,
        user_prompt
    )

# ============================================================
# AUDIO+TEXT RUN
# ============================================================

MODALITY = "audio_text"

rows = []

for idx, row in tqdm(
    merged_df.iterrows(),
    total=len(merged_df)
):

    wav_path = resolve_path(
        row["audio_path"]
    )

    if not os.path.exists(wav_path):

        print(f"⚠️ Missing audio: {wav_path}")

        continue

    transcript = row.get(
        "Utterance",
        ""
    )

    if pd.isna(transcript):

        print(f"⚠️ Missing transcript: {idx}")

        continue

    try:

        (
            system_prompt,
            base_user_prompt
        ) = build_fixed_prompt(
            row
        )

        final_user_prompt = (
            build_audio_text_prompt(
                transcript,
                base_user_prompt
            )
        )

        label, conf, raw = (
            judge_gemini_audio(
                wav_path,
                system_prompt,
                final_user_prompt,
                temperature=0.0
            )
        )
        print(system_prompt,final_user_prompt)

        rows.append({

            "id": row["id"],

            "audio_path": row["audio_path"],

            "labels": row["labels"],

            "language": row["language"],

            "transcript": transcript,

            "modality": MODALITY,

            "target": row["target"],

            "distractor": row["distractor"],

            "choices": row["choices"],

            "pred": label,

            "confidence": conf,

            "raw": raw,

            "model": MODEL_NAME
        })

    except Exception as e:

        print(f"❌ Error: {idx} | {e}")

        continue

    finally:

        cleanup()

    # ========================================================
    # INCREMENTAL SAVE
    # ========================================================

    if len(rows) >= 50:

        temp_df = pd.DataFrame(rows)

        if os.path.exists(OUTPUT_PATH):

            existing_df = pd.read_csv(
                OUTPUT_PATH
            )

            combined_df = pd.concat(
                [existing_df, temp_df],
                ignore_index=True
            )

            combined_df = combined_df.drop_duplicates(
                subset=["id"],
                keep="last"
            )

        else:

            combined_df = temp_df

        combined_df.to_csv(
            OUTPUT_PATH,
            index=False
        )

        print(
            f"Saved progress: {len(combined_df)}"
        )

        rows = []

# ============================================================
# FINAL SAVE
# ============================================================

final_df = pd.DataFrame(rows)

if os.path.exists(OUTPUT_PATH):

    existing_df = pd.read_csv(
        OUTPUT_PATH
    )

    final_df = pd.concat(
        [existing_df, final_df],
        ignore_index=True
    )

final_df = final_df.drop_duplicates(
    subset=["id"],
    keep="last"
)

final_df = final_df.sort_values(
    by="id"
).reset_index(drop=True)

final_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Final saved:", OUTPUT_PATH)

print("Final rows:", len(final_df))

In [ ]:
# ============================================================
# TRUE CLEAN RECOVERY PIPELINE (AUDIO + TEXT)
# ============================================================

TARGET_MODALITY = "audio_text"

# ============================================================
# PATHS
# ============================================================

SUBSET_PATH = os.path.join(
    BASE_DIR,
    "fixed_subset_1000.csv"
)

EXISTING_PATH = os.path.join(
    SAVE_DIR,
    "audio_text_results.csv"
)

# ============================================================
# LOAD TRUE SUBSET
# ============================================================

df_subset = pd.read_csv(SUBSET_PATH)

print("Expected subset size:", len(df_subset))

# ============================================================
# LOAD EXISTING RESULTS
# ============================================================

existing_df = pd.read_csv(EXISTING_PATH)

print("Original saved rows:", len(existing_df))

# ============================================================
# KEEP ONLY TARGET MODALITY
# ============================================================

existing_df = existing_df[
    existing_df["modality"] == TARGET_MODALITY
].copy()

print("After modality filtering:", len(existing_df))

# ============================================================
# REMOVE DUPLICATES
# ============================================================

existing_df = existing_df.drop_duplicates(
    subset=["id"],
    keep="first"
)

existing_df = existing_df.sort_values(
    by="id"
).reset_index(drop=True)

print("After dedup:", len(existing_df))

# ============================================================
# KEEP ONLY VALID ROWS
# ============================================================

valid_df = existing_df[

    existing_df["pred"].notna()

    & existing_df["confidence"].notna()

    & existing_df["raw"].notna()

].copy()

# remove empty-string rows too
valid_df = valid_df[

    (valid_df["pred"].astype(str).str.strip() != "")

    & (valid_df["raw"].astype(str).str.strip() != "")

]

print("Valid completed rows:", len(valid_df))

# ============================================================
# FIND TRUE MISSING IDS
# ============================================================

expected_ids = set(
    range(len(df_subset))
)

completed_ids = set(
    valid_df["id"].tolist()
)

missing_ids = sorted(
    list(expected_ids - completed_ids)
)

print(f"Missing IDs: {len(missing_ids)}")

print("First few missing IDs:")
print(missing_ids[:20])

# ============================================================
# RERUN MISSING ONLY
# ============================================================

rerun_rows = []

for idx in tqdm(missing_ids):

    row = df_subset.iloc[idx]

    wav_path = resolve_path(
        row["audio_path"]
    )

    if not os.path.exists(wav_path):
        print(f"⚠️ Missing audio: {wav_path}")
        continue

    labels = json.loads(
        row["labels"]
    )

    transcript = row.get(
        "transcript",
        ""
    )

    try:

        (
            system_prompt,
            base_user_prompt,
            target,
            distractor,
            choices
        ) = build_same_cluster_prompt(
            labels
        )

        final_user_prompt = (
            build_audio_text_prompt(
                transcript,
                base_user_prompt
            )
        )

        label, conf, raw = (
            judge_gemini_audio(
                wav_path,
                system_prompt,
                final_user_prompt,
                temperature=0.0
            )
        )

        # skip malformed outputs again
        if label is None:
            print(f"⚠️ Empty label: {idx}")
            continue

        rerun_rows.append({

            "id": idx,

            "audio_path": row["audio_path"],

            "labels": row["labels"],

            "language": row["language"],

            "modality": TARGET_MODALITY,

            "target": target,

            "distractor": distractor,

            "pred": label,

            "confidence": conf,

            "raw": raw,

            "model": MODEL_NAME
        })

    except Exception as e:

        print(f"❌ Failed again: {idx} | {e}")

        continue

    finally:

        cleanup()

# ============================================================
# MERGE CLEAN + RECOVERED
# ============================================================

rerun_df = pd.DataFrame(rerun_rows)

final_df = pd.concat(
    [valid_df, rerun_df],
    ignore_index=True
)

# deduplicate again just in case
final_df = final_df.drop_duplicates(
    subset=["id"],
    keep="first"
)

final_df = final_df.sort_values(
    by="id"
).reset_index(drop=True)

print("Final recovered rows:", len(final_df))

# ============================================================
# FINAL SAFETY CHECK
# ============================================================

final_missing = sorted(
    list(
        expected_ids
        - set(final_df["id"].tolist())
    )
)

print("Remaining missing:", len(final_missing))

if len(final_missing) > 0:

    print("Still missing IDs:")
    print(final_missing[:50])

# ============================================================
# SAVE CLEAN FILE
# ============================================================

RECOVERED_PATH = os.path.join(
    SAVE_DIR,
    f"{TARGET_MODALITY}_clean_recovered.csv"
)

final_df.to_csv(
    RECOVERED_PATH,
    index=False
)

print("Saved:", RECOVERED_PATH)